# OOD Main3 Consistency Ablation: Holdout-Environment OOD Results

Scenario filter: `holdout_env_ood`

This notebook reads the scenario-specific Main3 consistency ablation bundles and summarizes the saved best-family selections, environment-level results, and transfer heatmaps.


In [ ]:
from __future__ import annotations

import math
import os
import re
from collections import OrderedDict
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython import get_ipython
    from IPython.display import Markdown, display
except ImportError:  # pragma: no cover
    get_ipython = None
    Markdown = None

    def display(obj: Any) -> None:
        print(obj)


pd.options.display.max_columns = 200

RESULTS_ROOT = Path(
    os.environ.get(
        "OOD_MAIN3_SCENARIO_RESULTS_ROOT",
        "/playpen-ssd/smerrill/deception2/Results/OOD_Modeling_main3_consistency_xgb_pca_64_128_256",
    )
).expanduser().resolve()
SCENARIO_NAME = "holdout_env_ood"
SCENARIO_TITLE = "Train on 4 environments; evaluate OOD on the held-out environment"
REQUESTED_FEATURE_SIZES = [64, 128, 256]
ENV_ORDER = ["AdvisorAudit", "BS", "CarSales", "Gridworld", "Interview"]
TRAIN_AXIS_LABELS = (
    ENV_ORDER
    if SCENARIO_NAME == "single_source_ood"
    else [f"All except {env_name}" for env_name in ENV_ORDER]
)
TARGET_TITLES = OrderedDict(
    [
        ("delta_pos_gt_0_3", "delta_deception_rate > 0.3"),
        ("delta_neg_lt_neg_0_3", "delta_deception_rate < -0.3"),
    ]
)
MODEL_ORDER = ["GPT-OSS-20B", "Llama-8B", "Qwen-7B"]
FAMILY_ORDER = ["attention_only", "activation_only", "attention_plus_activation", "baseline"]
FAMILY_DISPLAY = {
    "attention_only": "Attention only",
    "activation_only": "Activation only",
    "attention_plus_activation": "Attention + activation",
    "baseline": "Baseline",
}
SHOW_HEATMAPS = True
SAVE_HEATMAPS = False
HEATMAP_EXPORT_DIR = RESULTS_ROOT / "notebook_exports" / SCENARIO_NAME


MODEL_NAME_OVERRIDES = {
    "gptoss20b": "GPT-OSS-20B",
    "gpt-oss-20b": "GPT-OSS-20B",
    "llama8b": "Llama-8B",
    "deepseek-r1-distill-llama-8b": "Llama-8B",
    "qwen7b": "Qwen-7B",
    "deepseek-r1-distill-qwen-7b": "Qwen-7B",
}


def md(text: str) -> None:
    shell_name = ""
    if get_ipython is not None and get_ipython() is not None:
        shell_name = get_ipython().__class__.__name__
    if Markdown is not None and shell_name == "ZMQInteractiveShell":
        display(Markdown(text))
    else:
        print(text)


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")


def canonical_model_name(model_dirname: object, fallback_name: str) -> str:
    if pd.notna(model_dirname) and str(model_dirname).strip():
        clean = slugify(str(model_dirname)).replace("_", "-")
        for key, value in MODEL_NAME_OVERRIDES.items():
            if key in clean:
                return value
        return str(model_dirname).strip()
    fallback = slugify(fallback_name).replace("_", "-")
    for key, value in MODEL_NAME_OVERRIDES.items():
        if key in fallback:
            return value
    return fallback_name


def read_config_map(path: Path) -> dict[str, str]:
    if not path.exists():
        return {}
    config_df = pd.read_csv(path)
    if {"setting", "value"}.issubset(config_df.columns):
        return {str(row.setting): str(row.value) for row in config_df.itertuples(index=False)}
    return {}


def maybe_read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def is_bundle_dir(path: Path) -> bool:
    return (path / "all_transfer_metrics.csv").exists()


def discover_bundle_dirs(root: Path) -> list[Path]:
    if is_bundle_dir(root):
        return [root.resolve()]
    if not root.exists():
        return []
    return sorted([path.resolve() for path in root.iterdir() if path.is_dir() and is_bundle_dir(path)], key=lambda path: path.name)


def model_sort_key(model_name: str) -> tuple[int, str]:
    try:
        return (MODEL_ORDER.index(model_name), model_name)
    except ValueError:
        return (len(MODEL_ORDER), model_name)


def family_sort_key(family_name: str) -> tuple[int, str]:
    try:
        return (FAMILY_ORDER.index(family_name), family_name)
    except ValueError:
        return (len(FAMILY_ORDER), family_name)


def env_sort_key(env_name: str) -> tuple[int, str]:
    try:
        return (ENV_ORDER.index(env_name), env_name)
    except ValueError:
        return (len(ENV_ORDER), env_name)


def load_scenario_bundle_frames(results_root: Path, scenario_name: str):
    inventory_rows: list[dict[str, object]] = []
    panel_frames: list[pd.DataFrame] = []
    model_frames: list[pd.DataFrame] = []
    metric_frames: list[pd.DataFrame] = []

    for bundle_dir in discover_bundle_dirs(results_root):
        config_map = read_config_map(bundle_dir / "config.csv")
        panel_df = maybe_read_csv(bundle_dir / "best_feature_space_by_target_size_family.csv")
        model_df = maybe_read_csv(bundle_dir / "best_model_by_target_size_family.csv")
        metrics_df = maybe_read_csv(bundle_dir / "all_transfer_metrics.csv")

        if panel_df.empty and model_df.empty and metrics_df.empty:
            continue

        bundle_scenarios: set[str] = set()
        for df in (panel_df, model_df, metrics_df):
            if not df.empty and "scenario_name" in df.columns:
                bundle_scenarios.update(df["scenario_name"].dropna().astype(str).tolist())
        if bundle_scenarios and scenario_name not in bundle_scenarios:
            continue

        model_name = canonical_model_name(config_map.get("model_dirname"), bundle_dir.name)
        bundle_id = str(bundle_dir.resolve())

        def _prepare(df: pd.DataFrame) -> pd.DataFrame:
            if df.empty:
                return df
            out = df.copy()
            if "scenario_name" in out.columns:
                out = out.loc[out["scenario_name"].astype(str).eq(scenario_name)].copy()
            if out.empty:
                return out
            out["bundle_id"] = bundle_id
            out["bundle_name"] = bundle_dir.name
            out["Model"] = model_name
            return out

        panel_df = _prepare(panel_df)
        model_df = _prepare(model_df)
        metrics_df = _prepare(metrics_df)

        if panel_df.empty and model_df.empty and metrics_df.empty:
            continue

        requested_sizes = []
        for df in (panel_df, model_df):
            if not df.empty and "requested_feature_size" in df.columns:
                requested_sizes.extend(pd.to_numeric(df["requested_feature_size"], errors="coerce").dropna().astype(int).tolist())
        requested_sizes = sorted(set(requested_sizes))

        inventory_rows.append(
            {
                "Model": model_name,
                "Bundle": bundle_dir.name,
                "Scenario": scenario_name,
                "Requested Sizes": ", ".join(str(size) for size in requested_sizes),
                "Path": bundle_id,
            }
        )
        if not panel_df.empty:
            panel_frames.append(panel_df)
        if not model_df.empty:
            model_frames.append(model_df)
        if not metrics_df.empty:
            metric_frames.append(metrics_df)

    inventory_df = pd.DataFrame(inventory_rows)
    if not inventory_df.empty:
        inventory_df = inventory_df.sort_values(["Model", "Bundle"], key=lambda col: col.map(lambda v: model_sort_key(v)[0] if col.name == "Model" else v)).reset_index(drop=True)

    panel_df = pd.concat(panel_frames, ignore_index=True) if panel_frames else pd.DataFrame()
    model_df = pd.concat(model_frames, ignore_index=True) if model_frames else pd.DataFrame()
    metrics_df = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
    return inventory_df, panel_df, model_df, metrics_df


inventory_df, panel_df, model_df, metrics_df = load_scenario_bundle_frames(RESULTS_ROOT, SCENARIO_NAME)


## Notes

- Each bundle is expected to live under a single shared results root and contain the saved CSV outputs from the scenario-aware ablation script.
- `requested_feature_size` refers to PCA dimensionality. Attention-only and baseline families are repeated across the requested-size panels because they do not use the PCA sweep directly.
- In the single-source notebook, diagonal heatmap cells are source-validation AUROC and off-diagonal cells are OOD AUROC.
- In the holdout notebook, each row is `All except <env>` and the held-out environment column is the OOD target for that row.


In [ ]:
def _clean_metric_values(values: pd.Series) -> pd.Series:
    return pd.to_numeric(values, errors="coerce").dropna()


def metric_mean(values: pd.Series) -> float:
    clean = _clean_metric_values(values)
    if clean.empty:
        return float("nan")
    return float(clean.mean())


def metric_se(values: pd.Series) -> float:
    clean = _clean_metric_values(values)
    if len(clean) <= 1:
        return float("nan")
    return float(clean.std(ddof=1) / math.sqrt(len(clean)))


def format_mean_se(values: pd.Series) -> str:
    mean_value = metric_mean(values)
    se_value = metric_se(values)
    if not np.isfinite(mean_value):
        return ""
    if np.isfinite(se_value):
        return f"{mean_value:.3f} +/- {se_value:.3f}"
    return f"{mean_value:.3f}"


def join_unique_text(values: pd.Series) -> str:
    ordered: list[str] = []
    seen: set[str] = set()
    for value in values.dropna().astype(str):
        clean = value.strip()
        if not clean or clean in seen:
            continue
        seen.add(clean)
        ordered.append(clean)
    return " | ".join(ordered)


def selected_panel_rows(
    panel_df: pd.DataFrame,
    model_df: pd.DataFrame,
    *,
    target_name: str,
    requested_feature_size: int,
) -> pd.DataFrame:
    panel_slice = panel_df.loc[
        panel_df["target_name"].astype(str).eq(target_name)
        & pd.to_numeric(panel_df["requested_feature_size"], errors="coerce").eq(int(requested_feature_size))
    ].copy()
    if panel_slice.empty:
        return pd.DataFrame()

    join_cols = [
        "bundle_id",
        "Model",
        "scenario_name",
        "target_name",
        "requested_feature_size",
        "requested_feature_size_label",
        "feature_family_group",
    ]
    extra_model_cols = [
        "feature_space",
        "feature_space_title",
        "feature_size",
        "feature_size_label",
        "train_env",
        "source_envs",
        "source_env_count",
        "heldout_env",
        "source_val_auroc",
        "mean_ood_auroc",
        "min_ood_auroc",
        "std_ood_auroc",
        "selected_features_path",
        "coefficients_path",
    ]
    model_slice = model_df.loc[
        model_df["target_name"].astype(str).eq(target_name)
        & pd.to_numeric(model_df["requested_feature_size"], errors="coerce").eq(int(requested_feature_size)),
        [column for column in join_cols + extra_model_cols if column in model_df.columns],
    ].copy()
    if not model_slice.empty:
        panel_slice = panel_slice.merge(
            model_slice,
            on=join_cols,
            how="left",
            validate="one_to_one",
            suffixes=("", "_best_model"),
        )

    panel_slice["Feature Family"] = panel_slice["feature_family_group"].map(FAMILY_DISPLAY).fillna(panel_slice["feature_family_group"])
    panel_slice = panel_slice.sort_values(
        ["Model", "feature_family_group"],
        key=lambda col: col.map(lambda value: model_sort_key(value)[0]) if col.name == "Model" else col.map(lambda value: family_sort_key(value)[0]),
    ).reset_index(drop=True)
    return panel_slice


def build_family_summary_table(
    panel_df: pd.DataFrame,
    model_df: pd.DataFrame,
    *,
    target_name: str,
    requested_feature_size: int,
) -> pd.DataFrame:
    selected_df = selected_panel_rows(
        panel_df,
        model_df,
        target_name=target_name,
        requested_feature_size=requested_feature_size,
    )
    if selected_df.empty:
        return pd.DataFrame()

    rows: list[dict[str, object]] = []
    for (model_name, family_name), family_df in selected_df.groupby(["Model", "feature_family_group"], dropna=False, sort=False):
        rows.append(
            {
                "Model": model_name,
                "Feature Family": FAMILY_DISPLAY.get(str(family_name), str(family_name)),
                "Selected Feature Set": join_unique_text(family_df["selected_feature_space_title"]),
                "Training Source(s)": join_unique_text(family_df["train_env"]),
                "Validation AUROC": format_mean_se(family_df["mean_val_auroc"]),
                "Mean OOD AUROC": format_mean_se(family_df["mean_ood_auroc"]),
                "Min OOD AUROC": format_mean_se(family_df["min_ood_auroc"]),
                "_model_sort": model_sort_key(str(model_name))[0],
                "_family_sort": family_sort_key(str(family_name))[0],
            }
        )
    out = pd.DataFrame(rows)
    return out.sort_values(["_model_sort", "_family_sort", "Model", "Feature Family"]).drop(columns=["_model_sort", "_family_sort"]).reset_index(drop=True)


def build_environment_summary_table(
    panel_df: pd.DataFrame,
    model_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    *,
    target_name: str,
    requested_feature_size: int,
) -> pd.DataFrame:
    selected_df = selected_panel_rows(
        panel_df,
        model_df,
        target_name=target_name,
        requested_feature_size=requested_feature_size,
    )
    if selected_df.empty or metrics_df.empty:
        return pd.DataFrame()

    metric_rows: list[dict[str, object]] = []
    for row in selected_df.itertuples(index=False):
        feature_space = getattr(row, "feature_space", getattr(row, "selected_feature_space", None))
        feature_size_label = getattr(row, "feature_size_label", getattr(row, "source_feature_size_label", None))
        train_env = getattr(row, "train_env", None)
        if feature_space is None or feature_size_label is None or train_env is None or pd.isna(train_env):
            continue
        subset = metrics_df.loc[
            metrics_df["bundle_id"].astype(str).eq(str(row.bundle_id))
            & metrics_df["Model"].astype(str).eq(str(row.Model))
            & metrics_df["target_name"].astype(str).eq(str(row.target_name))
            & metrics_df["feature_space"].astype(str).eq(str(feature_space))
            & metrics_df["feature_size_label"].astype(str).eq(str(feature_size_label))
            & metrics_df["train_env"].astype(str).eq(str(train_env))
        ].copy()
        if subset.empty:
            continue
        val_df = subset.loc[subset["eval_role"].astype(str).eq("val")].copy()
        ood_df = subset.loc[subset["eval_role"].astype(str).eq("ood")].copy()
        source_val_auroc = metric_mean(val_df["auroc"]) if not val_df.empty else float("nan")
        for environment, env_df in ood_df.groupby("test_env", dropna=False, sort=False):
            metric_rows.append(
                {
                    "Model": row.Model,
                    "feature_family_group": row.feature_family_group,
                    "Feature Family": FAMILY_DISPLAY.get(str(row.feature_family_group), str(row.feature_family_group)),
                    "Environment": str(environment),
                    "Selected Feature Set": getattr(row, "selected_feature_space_title", getattr(row, "feature_space_title", str(feature_space))),
                    "Training Source(s)": str(train_env),
                    "validation_auroc": source_val_auroc,
                    "ood_auroc": metric_mean(env_df["auroc"]),
                }
            )
    raw_df = pd.DataFrame(metric_rows)
    if raw_df.empty:
        return raw_df

    rows: list[dict[str, object]] = []
    for (model_name, family_name, environment), env_df in raw_df.groupby(["Model", "feature_family_group", "Environment"], dropna=False, sort=False):
        rows.append(
            {
                "Model": model_name,
                "Feature Family": FAMILY_DISPLAY.get(str(family_name), str(family_name)),
                "Environment": environment,
                "Selected Feature Set": join_unique_text(env_df["Selected Feature Set"]),
                "Training Source(s)": join_unique_text(env_df["Training Source(s)"]),
                "Validation AUROC": format_mean_se(env_df["validation_auroc"]),
                "OOD AUROC": format_mean_se(env_df["ood_auroc"]),
                "_model_sort": model_sort_key(str(model_name))[0],
                "_family_sort": family_sort_key(str(family_name))[0],
                "_env_sort": env_sort_key(str(environment))[0],
            }
        )
    out = pd.DataFrame(rows)
    return out.sort_values(["_model_sort", "_family_sort", "_env_sort", "Environment"]).drop(columns=["_model_sort", "_family_sort", "_env_sort"]).reset_index(drop=True)


def build_transfer_matrix_for_row(row: pd.Series, metrics_df: pd.DataFrame) -> pd.DataFrame:
    feature_space = row.get("feature_space", row.get("selected_feature_space"))
    feature_size_label = row.get("feature_size_label", row.get("source_feature_size_label"))
    train_env = row.get("train_env")
    subset = metrics_df.loc[
        metrics_df["bundle_id"].astype(str).eq(str(row["bundle_id"]))
        & metrics_df["Model"].astype(str).eq(str(row["Model"]))
        & metrics_df["target_name"].astype(str).eq(str(row["target_name"]))
        & metrics_df["feature_space"].astype(str).eq(str(feature_space))
        & metrics_df["feature_size_label"].astype(str).eq(str(feature_size_label))
        & metrics_df["train_env"].astype(str).eq(str(train_env))
    ].copy()
    matrix_df = pd.DataFrame(index=TRAIN_AXIS_LABELS, columns=ENV_ORDER, dtype=float)
    for metric_row in subset.itertuples(index=False):
        matrix_df.loc[str(metric_row.train_env), str(metric_row.test_env)] = float(metric_row.auroc) if pd.notna(metric_row.auroc) else float("nan")
    return matrix_df


def plot_selected_family_transfer_panels(
    panel_df: pd.DataFrame,
    model_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    *,
    model_name: str,
    target_name: str,
    requested_feature_size: int,
) -> None:
    selected_df = selected_panel_rows(
        panel_df,
        model_df,
        target_name=target_name,
        requested_feature_size=requested_feature_size,
    )
    selected_df = selected_df.loc[selected_df["Model"].astype(str).eq(model_name)].copy()
    if selected_df.empty:
        return
    selected_df = selected_df.sort_values("feature_family_group", key=lambda col: col.map(lambda value: family_sort_key(str(value))[0]))

    matrices: dict[str, pd.DataFrame] = {}
    finite_values: list[np.ndarray] = []
    for family_name in FAMILY_ORDER:
        family_rows = selected_df.loc[selected_df["feature_family_group"].astype(str).eq(family_name)]
        if family_rows.empty:
            continue
        row = family_rows.iloc[0]
        matrix_df = build_transfer_matrix_for_row(row, metrics_df)
        matrices[family_name] = matrix_df
        finite_values.append(matrix_df.to_numpy(dtype=float).ravel())

    flattened = np.concatenate(finite_values) if finite_values else np.array([], dtype=float)
    flattened = flattened[np.isfinite(flattened)]
    vmin = float(np.min(flattened)) if flattened.size else 0.0
    vmax = float(np.max(flattened)) if flattened.size else 1.0

    cmap = plt.cm.viridis.copy()
    cmap.set_bad(color="lightgray")

    fig, axes = plt.subplots(2, 2, figsize=(12.5, 10.0), constrained_layout=True)
    axes = np.asarray(axes).reshape(2, 2)
    image = None

    for idx, family_name in enumerate(FAMILY_ORDER):
        ax = axes.flat[idx]
        family_rows = selected_df.loc[selected_df["feature_family_group"].astype(str).eq(family_name)]
        if family_rows.empty or family_name not in matrices:
            ax.axis("off")
            continue
        row = family_rows.iloc[0]
        matrix_df = matrices[family_name].reindex(index=TRAIN_AXIS_LABELS, columns=ENV_ORDER)
        matrix = np.ma.masked_invalid(matrix_df.to_numpy(dtype=float))
        image = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xticks(np.arange(len(ENV_ORDER)))
        ax.set_xticklabels(ENV_ORDER, rotation=35, ha="right")
        ax.set_yticks(np.arange(len(TRAIN_AXIS_LABELS)))
        ax.set_yticklabels(TRAIN_AXIS_LABELS)
        ax.set_xlabel("Evaluation env")
        ax.set_ylabel("Training source(s)")
        ax.set_title(
            f"{FAMILY_DISPLAY.get(family_name, family_name)}\n"
            f"{row['selected_feature_space_title']}\n"
            f"train = {row['train_env']}",
            fontsize=10.0,
        )
        midpoint = (vmin + vmax) / 2.0 if np.isfinite(vmin) and np.isfinite(vmax) else 0.5
        for r in range(matrix_df.shape[0]):
            for c in range(matrix_df.shape[1]):
                value = matrix_df.iat[r, c]
                text = "nan" if not np.isfinite(value) else f"{value:.3f}"
                text_color = "white" if np.isfinite(value) and value < midpoint else "black"
                ax.text(c, r, text, ha="center", va="center", color=text_color, fontsize=8.0)

    if image is not None:
        fig.colorbar(image, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02, label="AUROC")

    scenario_note = (
        "diagonal = source validation AUROC; off-diagonal = OOD AUROC"
        if SCENARIO_NAME == "single_source_ood"
        else "rows show pooled-source validation on source envs and OOD on the held-out env"
    )
    fig.suptitle(
        f"{model_name} | {TARGET_TITLES[target_name]} | PCA size {requested_feature_size}\n"
        f"{scenario_note}",
        fontsize=14,
    )

    if SAVE_HEATMAPS:
        HEATMAP_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = HEATMAP_EXPORT_DIR / (
            f"{slugify(model_name)}__{target_name}__k{int(requested_feature_size):03d}__family_transfer_heatmaps.png"
        )
        fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()


if inventory_df.empty:
    md(
        f"_No populated bundle directories were found for scenario `{SCENARIO_NAME}` under `{RESULTS_ROOT}`._"
    )
else:
    available_sizes = sorted(
        set(pd.to_numeric(panel_df.get("requested_feature_size", pd.Series(dtype=float)), errors="coerce").dropna().astype(int).tolist())
    )
    requested_sizes = [size for size in REQUESTED_FEATURE_SIZES if size in available_sizes] or available_sizes

    md("## Bundle Inventory")
    display(inventory_df)

    md("## Scenario Summary")
    md(
        f"- Scenario: `{SCENARIO_NAME}`\n"
        f"- Description: {SCENARIO_TITLE}\n"
        f"- Results root: `{RESULTS_ROOT}`\n"
        f"- Requested PCA sizes present: {', '.join(str(size) for size in requested_sizes) if requested_sizes else 'none'}"
    )

    for target_name, target_title in TARGET_TITLES.items():
        md(f"## {target_title}")
        for requested_feature_size in requested_sizes:
            md(f"### PCA size {requested_feature_size}")
            family_summary_df = build_family_summary_table(
                panel_df,
                model_df,
                target_name=target_name,
                requested_feature_size=requested_feature_size,
            )
            if family_summary_df.empty:
                md("_No family-summary rows were found for this target / PCA size._")
            else:
                md("#### 1. Best Family Summary")
                display(family_summary_df)

            env_summary_df = build_environment_summary_table(
                panel_df,
                model_df,
                metrics_df,
                target_name=target_name,
                requested_feature_size=requested_feature_size,
            )
            if env_summary_df.empty:
                md("_No environment-summary rows were found for this target / PCA size._")
            else:
                md("#### 2. Best Family Summary By Evaluation Environment")
                display(env_summary_df)

            if SHOW_HEATMAPS and not family_summary_df.empty:
                present_models = [model_name for model_name in MODEL_ORDER if model_name in set(family_summary_df["Model"].astype(str))]
                for model_name in present_models:
                    md(f"#### 3. Transfer Heatmaps: {model_name}")
                    plot_selected_family_transfer_panels(
                        panel_df,
                        model_df,
                        metrics_df,
                        model_name=model_name,
                        target_name=target_name,
                        requested_feature_size=requested_feature_size,
                    )
